In [250]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import json
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib

In [251]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [252]:
df_train=pd.read_csv("../data/sent_train.csv")
df_valid=pd.read_csv("../data/sent_valid.csv")

In [253]:
print(df_train.isna().sum())
print(df_valid.isna().sum())

text     0
label    0
dtype: int64
text     0
label    0
dtype: int64


In [254]:
pd.set_option("display.max_colwidth", None)

In [255]:
print(df_train.duplicated().sum())
print(df_valid.duplicated().sum())


0
0


In [256]:
df_train.head()


,text,label
0,$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT,0
1,$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb",0
3,$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N,0
4,$FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB,0


In [257]:
df_valid.head()

,text,label
0,$ALLY - Ally Financial pulls outlook https://t.co/G9Zdi1boy5,0
1,"$DELL $HPE - Dell, HPE targets trimmed on compute headwinds https://t.co/YRUHZw7cYl",0
2,$PRTY - Moody's turns negative on Party City https://t.co/MBD5TFGC4P,0
3,$SAN: Deutsche Bank cuts to Hold,0
4,$SITC: Compass Point cuts to Sell,0


In [258]:
df_train.label.value_counts()

label
2    6178
1    1923
0    1442
Name: count, dtype: int64

In [259]:
df_valid.label.value_counts()

label
2    1566
1     475
0     347
Name: count, dtype: int64

In [260]:
def clean_texts(text):
    text=re.sub(r"https?://\S+","",text)
    text=re.sub(r'\$[A-Za-z]+\s*[,-|]?\s*',"",text)
    text=re.sub(r'<br\s*/?>','',text)
    text=re.sub(r'\s+', ' ', text).strip()
    return text

In [261]:
df_train["cleaned_text"]=df_train["text"].apply(clean_texts)
df_valid["cleaned_text"]=df_valid["text"].apply(clean_texts)

In [262]:
df_valid

,text,label,cleaned_text
0,$ALLY - Ally Financial pulls outlook https://t.co/G9Zdi1boy5,0,Ally Financial pulls outlook
1,"$DELL $HPE - Dell, HPE targets trimmed on compute headwinds https://t.co/YRUHZw7cYl",0,"Dell, HPE targets trimmed on compute headwinds"
2,$PRTY - Moody's turns negative on Party City https://t.co/MBD5TFGC4P,0,Moody's turns negative on Party City
3,$SAN: Deutsche Bank cuts to Hold,0,Deutsche Bank cuts to Hold
4,$SITC: Compass Point cuts to Sell,0,Compass Point cuts to Sell
...,...,...,...
2383,"Stocks making the biggest moves midday: TD Ameritrade, Tiffany, Uber, Hasbro & more https://t.co/gJM9nLZlzL",2,"Stocks making the biggest moves midday: TD Ameritrade, Tiffany, Uber, Hasbro & more"
2384,"Stocks making the biggest moves premarket: Fitbit, Xerox, Ford, Five Below, TripAdvisor & more",2,"Stocks making the biggest moves premarket: Fitbit, Xerox, Ford, Five Below, TripAdvisor & more"
2385,"Stocks making the biggest moves premarket: Home Depot, Kohl's, Disney, Broadcom & more",2,"Stocks making the biggest moves premarket: Home Depot, Kohl's, Disney, Broadcom & more"
2386,"Stocks making the biggest moves premarket: TD Ameritrade, Charles Schwab, Tesla & more https://t.co/gvK766Jmmm",2,"Stocks making the biggest moves premarket: TD Ameritrade, Charles Schwab, Tesla & more"


In [263]:
label_map={0:"negative", 1:"positive", 2:"neutral"}

In [264]:
df_train["sentiments"]=df_train["label"].map(label_map)
df_valid["sentiments"]=df_valid["label"].map(label_map)

In [265]:
df_train.head()

,text,label,cleaned_text,sentiments
0,$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT,0,JPMorgan reels in expectations on Beyond Meat,negative
1,$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3,0,Nomura points to bookings weakness at Carnival and Royal Caribbean,negative
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb",0,"Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook",negative
3,$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N,0,BTIG Research cuts to Neutral,negative
4,$FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB,0,Funko slides after Piper Jaffray PT cut,negative


In [266]:
df_valid.head()

,text,label,cleaned_text,sentiments
0,$ALLY - Ally Financial pulls outlook https://t.co/G9Zdi1boy5,0,Ally Financial pulls outlook,negative
1,"$DELL $HPE - Dell, HPE targets trimmed on compute headwinds https://t.co/YRUHZw7cYl",0,"Dell, HPE targets trimmed on compute headwinds",negative
2,$PRTY - Moody's turns negative on Party City https://t.co/MBD5TFGC4P,0,Moody's turns negative on Party City,negative
3,$SAN: Deutsche Bank cuts to Hold,0,Deutsche Bank cuts to Hold,negative
4,$SITC: Compass Point cuts to Sell,0,Compass Point cuts to Sell,negative


In [267]:
df_train[["cleaned_text", "sentiments"]].to_csv("../outputs/cleaned_data/cleaned_train.csv", index=False)
df_valid[["cleaned_text", "sentiments"]].to_csv("../outputs/cleaned_data/cleaned_valid.csv", index=False)

In [268]:
df_cleaned_train=pd.read_csv("../outputs/cleaned_data/cleaned_train.csv")
df_cleaned_valid=pd.read_csv("../outputs/cleaned_data/cleaned_valid.csv")


In [269]:
df_valid, df_test=train_test_split(
    df_cleaned_valid,
    test_size=0.5,
    random_state=42,
    stratify=df_valid["sentiments"]
)


In [270]:
for df in [df_cleaned_train, df_valid, df_test]:
    print("-------------------------")
    print(len(df))
    print(df["sentiments"].value_counts())

-------------------------
9543
sentiments
neutral     6178
positive    1923
negative    1442
Name: count, dtype: int64
-------------------------
1194
sentiments
neutral     783
positive    238
negative    173
Name: count, dtype: int64
-------------------------
1194
sentiments
neutral     783
positive    237
negative    174
Name: count, dtype: int64


#### Label Encoding

In [271]:

le=LabelEncoder()
df_cleaned_train["senti_label"]=le.fit_transform(df_cleaned_train.sentiments)
df_valid["senti_label"]=le.transform(df_valid.sentiments)
df_test["senti_label"]=le.transform(df_test.sentiments)

In [272]:
joblib.dump(
    le,
    "../outputs/encoders/label_encoder.pkl"
)

['../outputs/encoders/label_encoder.pkl']

In [273]:
#df_cleaned_train.dropna
print(df_cleaned_train.isna().sum())
print(df_valid.isna().sum())
print(df_test.isna().sum())

cleaned_text    11
sentiments       0
senti_label      0
dtype: int64
cleaned_text    1
sentiments      0
senti_label     0
dtype: int64
cleaned_text    2
sentiments      0
senti_label     0
dtype: int64


In [274]:
df_cleaned_train.dropna(inplace=True)
df_valid.dropna(inplace=True)
df_test.dropna(inplace=True)

In [275]:
#df_cleaned_train.dropna
print(df_cleaned_train.isna().sum())
print(df_valid.isna().sum())
print(df_test.isna().sum())

cleaned_text    0
sentiments      0
senti_label     0
dtype: int64
cleaned_text    0
sentiments      0
senti_label     0
dtype: int64
cleaned_text    0
sentiments      0
senti_label     0
dtype: int64


In [276]:
df_cleaned_train[["cleaned_text", "senti_label"]].to_csv("../outputs/cleaned_split_data/cleaned_split_train.csv", index=False)
df_valid[["cleaned_text", "senti_label"]].to_csv("../outputs/cleaned_split_data/cleaned_split_valid.csv",index=False)
df_test[["cleaned_text", "senti_label"]].to_csv("../outputs/cleaned_split_data/cleaned_split_test.csv", index=False)

In [277]:
df_cleaned_train=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_train.csv")
df_cleaned_valid=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_valid.csv")
df_cleaned_test=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_test.csv")

In [278]:
df_cleaned_train.head()

,cleaned_text,senti_label
0,JPMorgan reels in expectations on Beyond Meat,0
1,Nomura points to bookings weakness at Carnival and Royal Caribbean,0
2,"Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook",0
3,BTIG Research cuts to Neutral,0
4,Funko slides after Piper Jaffray PT cut,0


In [279]:
df_cleaned_valid.head()

,cleaned_text,senti_label
0,Monday's big rally is premature: financial advisor #economy #MarketScreener,1
1,Gold: Prepare For Bull Market To Begin Any Day. #business #finance #economy,2
2,Broadcom stock price target raised to $361 vs. $322 at SunTrust Robinson Humphrey,2
3,RECAP 12/10 +Pos Comments: + Hedgeye + Rosenblatt + CJS + BofAML,2
4,"Hurricane-force headwinds' pull oil lower, but the losses aren't built to last",0


In [280]:
df_cleaned_test.head()

,cleaned_text,senti_label
0,London Stock Exchange : Euronext Dublin Market Notice Replacement #LondonStockExchange #Stock #MarketScreener…,1
1,Does The Teradata Corporation (NYSE:TDC) Share Price Fall With The Market?,1
2,why macro funds are shutting down left and right,1
3,BDO Christmas party chaperones to guard against ghosts of scandals past,1
4,Enterprise Wins Favorable Ruling From Texas Supreme Court,2


#### Turn datasets into sentence embeddings

In [281]:
embedding_model=SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)
print(embedding_model.device)

cuda:0


In [282]:
X_train_embedding=embedding_model.encode(df_cleaned_train.cleaned_text.tolist(), batch_size=32, show_progress_bar=True)
X_valid_embedding=embedding_model.encode(df_cleaned_valid.cleaned_text.tolist(), batch_size=32, show_progress_bar=True)
X_test_embedding=embedding_model.encode(df_cleaned_test.cleaned_text.tolist(), batch_size=32, show_progress_bar=True)
print("EMbeddings are finished")

Batches: 100%|██████████| 38/38 [00:00<00:00, 157.33it/s]

EMbeddings are finished


In [283]:
print(X_train_embedding.shape)
print(X_valid_embedding.shape)
print(X_test_embedding.shape)

(9532, 384)
(1193, 384)
(1192, 384)


In [284]:
np.savez_compressed(
    "../outputs/embedded_data/train.npz",
    X=X_train_embedding,
    y=df_cleaned_train["senti_label"].to_numpy()
)

np.savez_compressed(
    "../outputs/embedded_data/valid.npz",
    X=X_valid_embedding,
    y=df_cleaned_valid["senti_label"].to_numpy()
)

np.savez_compressed(
    "../outputs/embedded_data/test.npz",
    X=X_test_embedding,
    y=df_cleaned_test["senti_label"].to_numpy()
)